# ตรวจจับไฟบนภาพนิ่ง (YOLO26 + Supervision)

<a href="https://colab.research.google.com/github/jakkzz/Fire-Detection-Drone/blob/main/th/Supervision_Image_Inferencing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🇹🇭 **ภาษาไทย** (เอกสารฉบับนี้) · [🇬🇧 English](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/Supervision_Image_Inferencing.ipynb)

รันโมเดลตรวจจับไฟที่เทรนไว้ใน [`drone_fire_detection_yolo26.ipynb`](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb)
กับภาพนิ่งหนึ่งภาพ แล้ววาดผลลัพธ์ด้วย [Supervision](https://supervision.roboflow.com/)

ไฟล์นี้ไม่จำเป็นต้องใช้ GPU เพราะ inference กับภาพเดียวรันบน CPU ก็ทันใจ

## 1. ติดตั้งไลบรารี

In [ ]:
%pip install -q "ultralytics>=8.4.122" "supervision>=0.30.0"

import supervision as sv
import ultralytics

print("ultralytics:", ultralytics.__version__)
print("supervision:", sv.__version__)

## 2. เตรียมไฟล์ weights และภาพทดสอบ

`best.pt` ไม่ได้เก็บไว้ในคลังโค้ด — ให้อัปโหลด checkpoint ที่ส่งออกมาจาก
[`drone_fire_detection_yolo26.ipynb`](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb)
ส่วนภาพตัวอย่างเซลล์ด้านล่างจะดึงมาจากคลังโค้ดให้เองอัตโนมัติ

In [ ]:
from pathlib import Path

MODEL_PATH = Path("best.pt")
IMAGE_PATH = Path("fire_image.png")

if not IMAGE_PATH.exists():
    !wget -q -O {IMAGE_PATH} https://raw.githubusercontent.com/jakkzz/Fire-Detection-Drone/main/fire_image.png

if not MODEL_PATH.exists():
    print("ไม่พบ best.pt — อัปโหลดไฟล์ตอนนี้ได้เลย (หรือกลับไปรันโน้ตบุ๊กสำหรับเทรนก่อน)")
    try:
        from google.colab import files  # type: ignore

        files.upload()
    except ImportError:
        raise FileNotFoundError("วาง best.pt ไว้ในโฟลเดอร์เดียวกับโน้ตบุ๊กนี้")

print("model:", MODEL_PATH.resolve())
print("image:", IMAGE_PATH.resolve())

## 3. รัน inference

In [ ]:
import cv2
from ultralytics import YOLO

model = YOLO(str(MODEL_PATH))

image = cv2.imread(str(IMAGE_PATH))
if image is None:
    raise FileNotFoundError(f"อ่านไฟล์ {IMAGE_PATH} ไม่ได้")

result = model(image, conf=0.25, verbose=False)[0]
detections = sv.Detections.from_ultralytics(result)

print("detections:", len(detections))

## 4. วาดกรอบและป้ายกำกับ

Supervision รุ่นปัจจุบันแยกการวาดออกเป็นหลาย annotator แล้ว `BoxAnnotator` ไม่รับ
อาร์กิวเมนต์ `labels=` อีกต่อไป (ถูกถอดออกตั้งแต่ supervision 0.22) ป้ายกำกับจึงเป็นหน้าที่ของ
`LabelAnnotator`

ชื่อชั้นข้อมูลอ่านมาจาก `detections["class_name"]` ซึ่ง `Detections.from_ultralytics`
เติมให้จากตัวโมเดลเอง จึงไม่ต้องมานั่งดูแลรายชื่อคลาสเองให้เสี่ยงหลุดไม่ตรงกับ weights ที่ใช้อยู่

In [ ]:
box_annotator = sv.BoxAnnotator(thickness=3)
label_annotator = sv.LabelAnnotator(text_scale=0.8, text_thickness=2, text_padding=6)

labels = [
    f"{class_name} {confidence:.2f}"
    for class_name, confidence
    in zip(detections["class_name"], detections.confidence)
]

annotated = box_annotator.annotate(scene=image.copy(), detections=detections)
annotated = label_annotator.annotate(scene=annotated, detections=detections, labels=labels)

# OpenCV อ่านภาพมาเป็น BGR ต้องแปลงก่อน สีใน matplotlib จึงจะถูกต้อง
sv.plot_image(image=cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB), size=(10, 10))

## 5. บันทึกภาพผลลัพธ์

In [ ]:
OUTPUT_PATH = Path("fire_image_annotated.png")
cv2.imwrite(str(OUTPUT_PATH), annotated)
print("saved:", OUTPUT_PATH.resolve())